# Log search and detection as code

Run one hunt in SPL, KQL and SQL over an online shop's logs, parse a raw log into fields, then turn the hunt into Sigma rules you test for hits and false positives, tune, and convert with pySigma.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `access.log`, `auth.log`, `eventlogs/Security.jsonl`, `lookbook_access.log`, `README.txt`.

## 1. How the search program works

Before you hunt, learn the program. A SIEM (security information and event management system) does four things with every log file it is given:

1. **Reads** each file and splits it into **events**, usually one line or one JSON record each.
2. **Stamps** every event with a time, `_time`, so events from different files line up on one clock.
3. **Recognises the format** and records it as the `sourcetype`: `syslog` for Linux logs, `access_combined` for web server logs, `json` for Windows events.
4. **Extracts fields** such as `src_ip`, `user`, `status` or `EventID`, so you can search on them instead of on raw text.

On screen you have a search box with a language switch (SPL or KQL), **Run**, a chart of **events over time**, the **field list** on the left (click a field to see its values and add one to your search), and the results as events or as a statistics table. **Help** lists the commands this teaching version knows.

The SIEM here is the company's: Chandni Textiles in Surat, a fabric shop that also sells online. Four logs from Monday 9 November have been loaded. Read `README.txt` first.

- [ ] Run the first search: what did the program make of each file?
- [ ] Run the second search: which programs wrote to auth.log?
- [ ] Click the field `src_ip` in the field list and read its values

> Hint: `*` means every event. `stats count by source sourcetype` makes one row per file and format, with the number of events in each.

```tool siem
{"q": "* | stats count by source sourcetype", "n": 4, "files": ["auth.log", "access.log", "lookbook_access.log", "eventlogs/Security.jsonl"], "blank": ["q"]}
```

```tool siem
{"q": "sourcetype=syslog | stats count by process", "n": 3, "files": ["auth.log"], "blank": ["q"]}
```

?> How many events does the program hold across the four files? :: sha256:3d3286f7cd19074f04e514b0c6c237e757513fb32820698b790e1dec801d947a
?> Which sourcetype did the program give auth.log? :: sha256:4d75f832ea2693e75fef081110af0c23a91d4d166a1ce1f63aa3d3d69cef9882

## 2. Start with the raw log

A hunt starts with a question, not a tool. Ours: **did anyone get into shop01, how did they do it, and what did they take?** You'll answer it three times: in SPL, in KQL and in SQL. Real teams use different SIEMs, and moving a hunt between them is a daily job.

The SIEM search here is a **teaching subset** of each language, so not every command works. When you want the real thing, the free [KC7](https://kc7cyber.com) game and the free Azure Data Explorer help cluster run full KQL.

First read the raw log with `grep`, so you know exactly what the search program is reading.

- [ ] How many lines does auth.log have?
- [ ] Which line types do you see: Failed, Accepted, sudo, CRON?
- [ ] Which address keeps failing, and which accounts did it try?

> Hint: `grep -c` counts matching lines. `awk '{print $(NF-3)}'` prints the fourth word from the end, which is the source address on every Failed line.

?> How many lines in auth.log contain "Failed password"? :: sha256:3fdba35f04dc8c462986c992bcf875546257113072a909c162f7e470e581e278

In [ ]:
%%bash
cd ~/case
wc -l auth.log access.log lookbook_access.log
grep -c 'Failed password' auth.log
grep 'Failed password' auth.log | awk '{print $(NF-3)}' | sort | uniq -c | sort -rn
grep 'Failed password' auth.log | awk '{print $(NF-5)}' | sort | uniq -c

## 3. SPL: count, then pivot

SPL reads left to right: a search, then a pipe `|`, then a command that works on the results. `stats count by src_ip` groups the failures by address, and `sort - count` puts the biggest first.

Then **pivot**: take what you found and search again from a new angle. First the address, then the account it reached, then what that account did.

- [ ] Run the failure count and read the table
- [ ] Find the Accepted login from the same address
- [ ] Find where that account normally logs in from
- [ ] Find what it did with sudo, and compare with its normal sudo use

> Hint: Field names come from the parsed log: `action`, `src_ip`, `user`, `auth_method`, `process`, `command`. Click any field name in the results to add it to the search.

```tool siem
{"q": "sourcetype=syslog action=failure | stats count by src_ip | sort - count", "n": 2, "files": ["auth.log"], "blank": ["q"]}
```

```tool siem
{"q": "Accepted src_ip=203.0.113.45 | table _time user auth_method", "n": 1, "files": ["auth.log"], "blank": ["q"]}
```

```tool siem
{"q": "process=sshd user=catalog | table _time src_ip action", "n": 5, "files": ["auth.log"], "blank": ["q"]}
```

```tool siem
{"q": "process=sudo | table _time user run_as command", "n": 3, "files": ["auth.log"], "blank": ["q"]}
```

?> Which address failed most often? :: sha256:eb4b2dd34d5ba9aff5f39480c21f10aff75f58d1bfe56c0d153cb60b63533992
?> Which account did that address log in as? :: sha256:652f55016243bf1b9f1bbea46d5749ef892dbe394e46de9d66ab1aacf0b4af57
?> From which address does that account normally log in? :: sha256:274d999757a71abe0670ab2d0168387ab973af5c51acb41c3cbe3f79d4221453
?> Which program did the attacker run with sudo to copy every database? (program name) :: sha256:2dd1a3c0c997e45718e7fb77a1089c6f60c5b222875f92fdeca2a53ae361452d

## 4. The same hunt in KQL

KQL starts with a **table**, then pipes into operators: `where` filters, `summarize count() by` groups, `project` picks columns, `sort by` orders. The table is named after the file: `auth` for auth.log. The answer must be the same as in SPL. If it isn't, one of the two searches is wrong.

- [ ] Run the failure count in KQL and compare with SPL
- [ ] Write the success pivot in KQL
- [ ] Note one thing that is easier in each language

> Hint: In KQL the count column is called `count_`. Strings use `==` and double quotes.

```tool siem
{"q": "auth | where action == \"failure\" | summarize count() by src_ip | sort by count_ desc", "lang": "kql", "n": 2, "files": ["auth.log"], "blank": ["q"]}
```

```tool siem
{"q": "auth | where src_ip == \"203.0.113.45\" and action == \"success\" | project _time, user, auth_method", "lang": "kql", "n": 1, "files": ["auth.log"], "blank": ["q"]}
```

?> How many failed logins came from that address? (KQL count_) :: sha256:4fc82b26aecb47d2868c4efbe3581732a3e7cbcc6c2efb32062c08170a05eeb8

## 5. SQL over the same logs

Every log is also a table. The SQL tool names each table after its file (`auth`, `access`, `security`) and makes a column for each field. SQL is the language behind most data lakes, so it is worth having in your pocket.

- [ ] Count requests per client address in the shop's web log
- [ ] Find every request with UNION in it
- [ ] Join the idea back to the SSH hunt: same address?

> Hint: `GROUP BY clientip ORDER BY hits DESC` is the SQL version of `stats count by clientip | sort - count`. `LIKE '%UNION%'` finds the word anywhere in a column.

```tool sql
{"q": "SELECT clientip, COUNT(*) AS hits FROM access GROUP BY clientip ORDER BY hits DESC", "n": 10, "files": ["access.log"], "blank": ["q"]}
```

```tool sql
{"q": "SELECT clientip, status, uri FROM access WHERE uri LIKE '%UNION%'", "n": 3, "files": ["access.log"], "blank": ["q"]}
```

?> How many requests did 203.0.113.45 make to the shop? :: sha256:6b51d431df5d7f141cbececcf79edf3dd861c3b4069f0b11661a3eefacbba918

## 6. Parse a raw log into fields

Searches only work on fields someone parsed. When a log arrives that your SIEM doesn't know, you write the parser. The log parser uses `%{TYPE:name}` pieces, the same idea as Grok in log pipelines.

The pattern below parses SSH login lines. Lines it can't parse are shown in red: decide whether that is right.

- [ ] Run the pattern on auth.log
- [ ] Which lines don't parse, and why is that fine?
- [ ] Write a second pattern for the sudo lines

> Hint: The sudo line has a different shape: `sudo:  catalog : TTY=... ; COMMAND=...`. Start with `%{SYSLOGTIMESTAMP:time} %{HOSTNAME:host} sudo: +%{USER:user} : %{GREEDYDATA:rest}`.

```tool parse
{"file": "auth.log", "pat": "%{SYSLOGTIMESTAMP:time} %{HOSTNAME:host} sshd\\[%{INT:pid}\\]: %{WORD:result} %{WORD:method} for (?:invalid user )?%{USER:user} from %{IP:src_ip} port %{INT:src_port}%{GREEDYDATA:rest}", "n": 16, "blank": ["pat"]}
```

?> How many of the 26 lines does the SSH pattern parse? :: sha256:b17ef6d19c7a5b1ee83b907c595526dcb1eb06db8227d650d5dda0a9f4ce8cd9

## 7. Hunt the web attack in SPL and KQL

The SSH password guessing failed. So how did the attacker have catalog's password 40 minutes later? Look at the web log. `uri_path` is the page without the query string, which makes grouping easy.

- [ ] Which tool did the attacker use? (look at the user agent)
- [ ] Which database table did the injection read?
- [ ] What did they do in the admin pages, and how big was it?
- [ ] Write both searches into your hunt pack

> Hint: `%20` is a space, `%27` is a quote and `%3D` is `=` in a URL. CyberChef's URL Decode makes the requests readable.

```tool siem
{"q": "sourcetype=access_combined useragent=sqlmap* | table _time clientip uri status", "n": 7, "files": ["access.log"], "blank": ["q"]}
```

```tool siem
{"q": "sourcetype=access_combined clientip=203.0.113.45 uri_path=/admin* | table _time method uri status bytes", "n": 4, "files": ["access.log"], "blank": ["q"]}
```

```tool siem
{"q": "access | where clientip == \"203.0.113.45\" and uri_path startswith \"/admin\" | project _time, method, uri, status, bytes", "lang": "kql", "n": 4, "files": ["access.log"], "blank": ["q"]}
```

?> Which scanner's name is in the user agent? :: sha256:44b5a25dfad00357b11b2515f85217ecbcc302794bfa6d4fc888fa0a45cecb07
?> Which table held the staff passwords? :: sha256:223470ff416af9d3fb0a1b445a10e1a8824cd9f63fe27704e68043b1f3ae4cd8
?> Which admin page did the attacker download? (the path, without ?from=...) :: sha256:9c5138acf5a4eb4454419859930a846811da7642e8dd783c7f92490492ee2278

## 8. From hunt to rule: your first Sigma rule

A hunt you run once finds one attack. A **rule** finds the next one. Sigma is the open format for detection rules: one YAML file that converts to SPL, KQL and others.

Open the Windows log from accts-pc02 (`eventlogs/Security.jsonl`). On Monday morning Excel opened `Payment_Advice_09Nov.xlsm` from Outlook and started cmd.exe. The rule below fires on that, and on Word or Outlook doing the same. Press **Test**, then **Convert with pySigma** to see the real converter's SPL.

- [ ] Test the rule: how many events match?
- [ ] Convert it with pySigma and compare with a search you would write by hand
- [ ] Excel also started `splwow64.exe` that morning. Why doesn't that match, and why is that right?
- [ ] Why does PowerShell not match? (look at its parent)

> Hint: splwow64.exe is the Windows printing helper: Excel starts it whenever someone prints. PowerShell's parent was cmd.exe, not Excel: one rule per step of the chain, or a second rule for "cmd.exe starts PowerShell hidden".

```tool siem
{"q": "EventID=4688 | stats count by Computer", "n": 1, "files": ["eventlogs/Security.jsonl"], "blank": ["q"]}
```

```tool sigma
{"rule": "title: Office program starts a command shell\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000001\nstatus: experimental\ndescription: Word, Excel or Outlook starting cmd.exe or PowerShell is how most macro malware begins.\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4688\n    ParentProcessName|endswith:\n      - '\\WINWORD.EXE'\n      - '\\EXCEL.EXE'\n      - '\\OUTLOOK.EXE'\n    NewProcessName|endswith:\n      - '\\cmd.exe'\n      - '\\powershell.exe'\n  condition: selection\nfalsepositives:\n  - Office add-ins that call a script on purpose (rare; list them)\nlevel: high\ntags:\n  - attack.execution\n  - attack.t1204.002\n", "n": 1, "ls": true, "blank": ["rule"]}
```

?> How many process-creation events (4688) are in Security.jsonl? :: sha256:4523540f1504cd17100c4835e85b7eefd49911580f8efff0599a8f283be6b9e3
?> How many events does the Office-starts-a-shell rule match? :: sha256:6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b

## 9. Test for false positives, then tune

A rule that fires on everything is worse than no rule: people stop reading it. Test every rule twice: does it catch the attack (**true positives**) and does it stay quiet on normal work (**false positives**)?

The first rule below is deliberately noisy. Tick the events that are really malicious; the tool scores precision. Then use the tuned rules, which each describe one thing an attacker did.

- [ ] Run the noisy rule and tick the malicious events
- [ ] Run each tuned rule: each should match only attack events
- [ ] Write one tuning note per rule: what it misses, and what would make it noisy

> Hint: Good rules describe behaviour that normal users don't have: a program in C:\Users\Public, shadow copies deleted, a new member of Administrators.

```tool sigma
{"rule": "title: Any new process on a workstation\nstatus: experimental\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4688\n  condition: selection\nlevel: low\n", "n": 17, "ls": true, "blank": ["rule"]}
```

```tool sigma
{"rule": "title: Program started from C:\\Users\\Public\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000002\nstatus: experimental\ndescription: Programs should not run from the Public profile, which every user can write to.\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4688\n    NewProcessName|contains: '\\Users\\Public\\'\n  condition: selection\nfalsepositives:\n  - Badly packaged software installed to the Public profile (check with IT)\nlevel: high\ntags:\n  - attack.defense_evasion\n  - attack.t1036.005\n", "n": 2, "ls": true, "blank": ["rule"]}
```

```tool sigma
{"rule": "title: Shadow copies deleted\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000003\nstatus: experimental\ndescription: Ransomware deletes Volume Shadow Copies so files can't be restored.\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4688\n    CommandLine|contains|all:\n      - 'vssadmin'\n      - 'delete shadows'\n  condition: selection\nfalsepositives:\n  - Backup software clearing old copies (it rarely uses /all /quiet)\nlevel: critical\ntags:\n  - attack.impact\n  - attack.t1490\n", "n": 1, "ls": true, "blank": ["rule"]}
```

```tool sigma
{"rule": "title: Account added to the local Administrators group\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000004\nstatus: experimental\ndescription: A new member of Administrators on a workstation needs a change ticket.\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4732\n    TargetUserName: 'Administrators'\n  condition: selection\nfalsepositives:\n  - IT adding a helpdesk account during a planned change\nlevel: high\ntags:\n  - attack.persistence\n  - attack.t1136.001\n", "n": 1, "ls": true, "blank": ["rule"]}
```

?> How many events does the noisy "any new process" rule match? :: sha256:4523540f1504cd17100c4835e85b7eefd49911580f8efff0599a8f283be6b9e3
?> Which account was added to Administrators? :: sha256:97d686e2e855f8857ab5ad18a6a2b4637fa9180b51f4b43a3f1e15a01fa3af9c
?> Which program deleted the shadow copies? (program name) :: sha256:ff80268dfba624cd94529df29c7daed2f4ddd732fa3317dea937d41e10feb99a

## 10. A web rule, tested on two web logs

Sigma works for web logs too. The first version below matches any `%27` (a quote), the word UNION, or the tool's name. `lookbook_access.log` is an hour of normal visitors to the company's lookbook site: see who the noisy version accuses there. The tuned version keys on the tool's name and on `UNION%20SELECT`.

- [ ] Test both versions on both web logs
- [ ] Which requests does the noisy version catch that the tuned one doesn't? Are they attacks?
- [ ] One attack request did not come from sqlmap. Which part of the tuned rule catches it?
- [ ] Write the tuning note: when would the tuned rule miss an attack?

> Hint: An attacker can change the user agent in one flag, and here they did, by pasting a payload into a browser. That's why the rule keeps a second selection for the injection itself.

```tool sigma
{"rule": "title: SQL injection in a web request\nstatus: experimental\nlogsource:\n  category: webserver\ndetection:\n  keywords:\n    - '%27'\n    - 'UNION'\n    - 'sqlmap'\n  condition: keywords\nlevel: high\n", "n": 10, "ls": true, "blank": ["rule"]}
```

```tool sigma
{"rule": "title: SQL injection tool or UNION SELECT in a web request\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000005\nstatus: experimental\nlogsource:\n  category: webserver\ndetection:\n  tool:\n    - 'sqlmap/'\n  union:\n    - 'UNION%20SELECT'\n  condition: tool or union\nfalsepositives:\n  - Your own authorised scans (exclude the scanner's address in the tuning notes)\nlevel: high\ntags:\n  - attack.initial_access\n  - attack.t1190\n", "n": 8, "ls": true, "blank": ["rule"]}
```

?> How many web requests does the noisy version match? :: sha256:4a44dc15364204a80fe80e9039455cc1608281820fe2b24f1e5233ade6af1dd5
?> How many web requests does the tuned rule match? :: sha256:2c624232cdd221771294dfbb310aca000a0df6ac8b66b696d90ef06fdefb64a3

## 11. Hand in your hunt pack and rule pack

Your hand-in is this notebook, with every search and rule saved in it. Before you hand in:

| Piece | What it must have |
|---|---|
| Hunt pack | The SSH hunt and the web hunt, each in SPL and KQL, with the answer they give |
| SQL | One grouping query and one sentence on what it shows |
| Rule pack | Five Sigma rules (four Windows, one web), each tested, with a tuning note |
| Conversion | Each rule converted with pySigma, and one difference you noticed |
| The story | Three sentences for Chandni Textiles: how the attacker got in, what they took, and the first thing to fix |

- [ ] Every question answered
- [ ] Every rule shows its test result
- [ ] A tuning note under each rule
- [ ] Press Hand in

**Rubric**

| Criterion | Points |
|---|---|
| Hunt pack: the SSH hunt in SPL and in KQL, both giving the same answer, with the account's normal source compared | 4 |
| Hunt pack: the web attack hunt in SPL and KQL, with the attacker's IP, the table they read and what they downloaded | 4 |
| SQL: one query that groups and counts, with the result explained | 2 |
| Sigma: four Windows rules that each match the attack and nothing else | 6 |
| Sigma: the web rule, with its false-positive test and tuning note | 3 |
| Conversion: each rule converted with pySigma, with one difference from your hand-written search noted | 3 |
| Tuning notes: what each rule misses, and what would make it noisy | 3 |